## **Importing Necessary Libraries**

In [ ]:
import re
import nltk
import spacy
import string
import pandas as pd
from tqdm import tqdm # high-quality progress bars
from nltk.corpus import stopwords

In [ ]:
nltk.download('stopwords')
nlp = spacy.load("en_core_web_sm") # English core language

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\LOQ\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


### **Specify Stopwords Language**

In [ ]:
stop_words = set(stopwords.words("english"))

In [7]:
tqdm.pandas()

## **Load the Dataset**

In [9]:
fake_path = r"E:\Arshdeep CL NLP Project\Dataset\raw\Fake.csv"
real_path = r"E:\Arshdeep CL NLP Project\Dataset\raw\True.csv"
fake_df = pd.read_csv(fake_path)
real_df = pd.read_csv(real_path)

### **Adding labels: 0 = Fake, 1 = Real**

In [15]:
fake_df["label"] = 0
real_df["label"] = 1

In [16]:
df = pd.concat([fake_df, real_df], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

In [17]:
df.head()

,title,text,subject,date,label
0,Ben Stein Calls Out 9th Circuit Court: Committ...,"21st Century Wire says Ben Stein, reputable pr...",US_News,"February 13, 2017",0
1,Trump drops Steve Bannon from National Securit...,WASHINGTON (Reuters) - U.S. President Donald T...,politicsNews,"April 5, 2017",1
2,Puerto Rico expects U.S. to lift Jones Act shi...,(Reuters) - Puerto Rico Governor Ricardo Rosse...,politicsNews,"September 27, 2017",1
3,OOPS: Trump Just Accidentally Confirmed He Le...,"On Monday, Donald Trump once again embarrassed...",News,"May 22, 2017",0
4,Donald Trump heads for Scotland to reopen a go...,"GLASGOW, Scotland (Reuters) - Most U.S. presid...",politicsNews,"June 24, 2016",1


In [19]:
df.to_csv("E:\Arshdeep CL NLP Project\Dataset/preprocessed/combined_labeled.csv", index=False)

## **Text Cleaning**

In [21]:
def clean_text(text):
    text = str(text)
    text = text.lower()  # Lowercase
    text = re.sub(r"http\S+", "", text)  # Remove URLs
    text = re.sub(r"[^a-zA-Z\s]", "", text)  # Remove special characters, numbers
    text = re.sub(r"\s+", " ", text)  # Remove extra spaces
    return text.strip()

## **Lemmatisation with spaCy**

In [20]:
def lemmatize_text(text):
    doc = nlp(text)
    return " ".join(
        [token.lemma_ for token in doc if token.text not in stop_words and token.text not in string.punctuation]
    )

In [22]:
def lemmatize_spacy_pipe(texts):
    cleaned_texts = []
    docs = nlp.pipe(texts, batch_size=64, n_process=-1)  # use all cores

    for doc in docs:
        tokens = [
            token.lemma_
            for token in doc
            if token.text not in stop_words and not token.is_punct
        ]
        cleaned_texts.append(" ".join(tokens))

    return cleaned_texts

## **Cleaning + Lemmatisation**

In [24]:
df["clean_text"] = df["text"].apply(clean_text)

df["lemmatized_text"] = lemmatize_spacy_pipe(df["clean_text"].tolist())

## **Saving the Cleaned Dataset**

In [25]:
df.to_csv("E:\Arshdeep CL NLP Project\Dataset\preprocessed/fake_news_cleaned.csv", index=False)
print("Saved cleaned data!")

Saved cleaned data!
